# 서울시 자치구별 폐기물 배출량 회귀분석

클러스터링 결과(자치구 유형)를 독립변수로 활용하여, 서울 자치구별 **총 생활폐기물 배출량**을 설명하는 회귀모형을 단계적으로 구축한다.  
최종적으로 **기대 배출량 대비 초과/미달 자치구**를 식별하여 정책적 시사점을 도출한다.

**종속변수**: `log(waste_total)` (로그 변환된 총 폐기물 배출량)  
**추정 방법**: OLS + 자치구 클러스터 robust standard error

## 0. 라이브러리 로드

In [1]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

## 1. 데이터 불러오기

- `eda_master_v2_by_gu_year.csv`: 자치구×연도별 폐기물, 인구, 사업체 등 구조변수
- `cluster_result.csv`: 클러스터링 노트북에서 생성한 자치구 유형 정보

두 데이터를 자치구(`gu`) 기준으로 병합한다.

In [2]:
df_minseo = pd.read_csv("C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/regresssion/eda_master_v2_by_gu_year.csv")
df_cluster = pd.read_csv("C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/clustering/cluster_result.csv")
df_minseo

,year,gu,waste_total,waste_per_capita,food_waste_total,recycle_rate,total_pop,households,elderly_ratio,parcel_arrival_total,arrival_per_hh,living_pop_daily_avg,day_night_ratio,biz_total,food_accom_biz,illegal_dumping_count
0,2020,강남구,254317,694.9,242.6,65.553620,544055,234872,13.803016,NaN,NaN,1.932652e+07,1.328122,115054,3816,22467
1,2020,강동구,134696,368.0,110.4,67.471194,463998,196499,15.090367,NaN,NaN,1.210167e+07,0.915846,40978,4897,8282
2,2020,강북구,88989,243.1,68.7,60.028768,311569,145896,20.355684,NaN,NaN,7.291377e+06,0.897156,26711,2994,9747
3,2020,강서구,192797,526.8,144.1,69.431060,585901,266982,15.187549,NaN,NaN,1.300812e+07,0.948083,58788,8490,362
4,2020,관악구,146977,401.6,94.2,70.433469,509803,274811,15.471663,NaN,NaN,1.169594e+07,0.881875,38639,4050,6151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2023,용산구,208742,571.9,63.2,82.395014,227106,107825,17.501519,3634526.0,33.707637,6.765582e+06,1.160108,28363,1701,645
96,2023,은평구,123761,339.1,72.6,59.823369,470869,215721,19.924225,5192988.0,24.072705,1.006468e+07,0.855890,36201,4716,7025
97,2023,종로구,97419,266.9,79.9,56.079410,150453,72067,19.118263,2677305.0,37.150221,6.708106e+06,1.548435,46977,1916,20268
98,2023,중구,132481,363.0,101.7,59.428144,131793,64714,19.667205,3291386.0,50.860494,6.341925e+06,1.856137,67202,2325,9012


In [3]:
df_cluster

,gu,cluster_anchor,cluster_type,L1_work_home_ratio,L2_weekend_active_ratio,weekend_home_ratio,B1_avg_emp_per_biz,B2_office_emp_ratio_no_support,B3_consumption_tourism_emp_ratio,B4_life_service_emp_ratio,B5_industrial_logistics_emp_ratio,PC1_도시기능,PC2_도시기능
0,종로구,0,상업형,1.639698,0.795812,1.120581,5.672325,0.307603,0.270190,0.158138,0.132324,3.458148,1.960962
1,중구,0,상업형,2.064828,0.629787,1.083157,5.960833,0.410461,0.276900,0.074487,0.119097,5.194560,1.344587
2,용산구,0,상업형,1.184049,0.975406,1.076227,5.656782,0.311226,0.344603,0.141911,0.084112,1.909782,0.504961
3,성동구,3,혼합형,1.092607,0.911819,0.982540,5.119779,0.264219,0.266925,0.150575,0.206280,0.829732,-0.715104
4,광진구,2,유동집중형,0.937800,1.047381,0.982846,3.768584,0.143997,0.298187,0.273648,0.181917,-1.289669,0.229185
5,동대문구,2,유동집중형,0.982189,0.977135,0.971601,3.378678,0.124829,0.317755,0.284616,0.179222,-1.331756,0.346242
6,중랑구,2,유동집중형,0.847549,1.097254,0.967456,2.727116,0.076985,0.272712,0.268140,0.302640,-2.305739,0.312982
7,성북구,1,주거형,0.897531,1.021568,0.951521,3.531200,0.122407,0.262859,0.367205,0.155216,-2.065834,0.189028
8,강북구,2,유동집중형,0.873287,1.080638,0.976905,2.953737,0.107578,0.302779,0.294023,0.199744,-2.043262,0.443315
9,도봉구,1,주거형,0.866140,1.096668,0.978875,3.165744,0.118359,0.249941,0.315619,0.194613,-2.066692,0.451756


### 1-1. 데이터 병합

자치구명의 공백을 제거한 뒤, `cluster_type`(상업형/주거형/유동집중형/혼합형)을 회귀 데이터에 병합한다.  
결측이 없는지 확인한다.

In [4]:
# 복사본 생성
reg = df_minseo.copy()
clu = df_cluster.copy()

# 자치구명 공백 제거
reg["gu"] = reg["gu"].astype(str).str.strip()
clu["gu"] = clu["gu"].astype(str).str.strip()

# 회귀에는 우선 cluster_type만 사용
# cluster_anchor는 숫자라서 같이 보관만 해도 됨
cluster_use = clu[["gu", "cluster_anchor", "cluster_type"]].drop_duplicates()

# merge
df = reg.merge(cluster_use, on="gu", how="left")

print(df.shape)
print("cluster_type 결측 수:", df["cluster_type"].isna().sum())

df[["gu", "cluster_anchor", "cluster_type"]].drop_duplicates().sort_values("gu")

(100, 18)
cluster_type 결측 수: 0


,gu,cluster_anchor,cluster_type
0,강남구,0,상업형
1,강동구,2,유동집중형
2,강북구,2,유동집중형
3,강서구,3,혼합형
4,관악구,1,주거형
5,광진구,2,유동집중형
6,구로구,3,혼합형
7,금천구,3,혼합형
8,노원구,1,주거형
9,도봉구,1,주거형


## 2. 파생변수 생성

회귀모형에 사용할 변수를 가공한다.

| 변수 | 산식 | 의미 |
|------|------|------|
| `log_waste_total` | log(waste_total) | 종속변수 (로그 변환) |
| `log_total_pop` | log(total_pop) | 주민등록인구 (규모 통제) |
| `log_living_pop` | log(living_pop_daily_avg) | 일평균 생활인구 (유동인구 효과) |
| `biz_per_10k` | biz_total / total_pop × 10,000 | 인구 1만명당 사업체 수 |
| `food_accom_ratio` | food_accom_biz / biz_total | 음식·숙박업 비율 |
| `illegal_per_10k` | illegal_dumping_count / total_pop × 10,000 | 인구 1만명당 불법투기 건수 |

로그 변환은 **우측 꼬리가 긴 분포**를 정규화하고, 계수를 **탄력성(%)** 으로 해석할 수 있게 한다.

In [5]:
# 파생변수 만들기 
df_model = df.copy()

# 종속변수
df_model["log_waste_total"] = np.log(df_model["waste_total"])

# 규모 변수 로그
df_model["log_total_pop"] = np.log(df_model["total_pop"])
df_model["log_living_pop"] = np.log(df_model["living_pop_daily_avg"])

# 구조 변수
df_model["biz_per_10k"] = df_model["biz_total"] / df_model["total_pop"] * 10000
df_model["food_accom_ratio"] = df_model["food_accom_biz"] / df_model["biz_total"]
df_model["illegal_per_10k"] = df_model["illegal_dumping_count"] / df_model["total_pop"] * 10000

# 범주형 처리
df_model["year"] = df_model["year"].astype(int)
df_model["cluster_type"] = df_model["cluster_type"].astype("category")

# 확인
df_model[[
    "year", "gu", "cluster_type",
    "log_waste_total", "log_total_pop", "log_living_pop",
    "elderly_ratio", "day_night_ratio", "biz_per_10k",
    "food_accom_ratio", "illegal_per_10k"
]].head()

,year,gu,cluster_type,log_waste_total,log_total_pop,log_living_pop,elderly_ratio,day_night_ratio,biz_per_10k,food_accom_ratio,illegal_per_10k
0,2020,강남구,상업형,12.446337,13.206806,16.776989,13.803016,1.328122,2114.749428,0.033167,412.954573
1,2020,강동구,유동집중형,11.810776,13.047636,16.308854,15.090367,0.915846,883.150358,0.119503,178.492149
2,2020,강북구,유동집중형,11.396268,12.649376,15.802203,20.355684,0.897156,857.306086,0.112089,312.836001
3,2020,강서구,혼합형,12.169393,13.280906,16.381085,15.187549,0.948083,1003.377704,0.144417,6.178518
4,2020,관악구,주거형,11.898031,13.141780,16.274752,15.471663,0.881875,757.920216,0.104816,120.654449


## 3. 표준화 (Z-score)

연속형 독립변수를 Z-score로 표준화하여 **계수 크기를 직접 비교** 가능하게 한다.  
표준화된 변수명에는 `_z` 접미사를 붙인다.

In [6]:
#표준화 
model_cols = [
    "year", "gu", "cluster_type",
    "waste_total", 
    "log_waste_total",
    "log_total_pop",
    "log_living_pop",
    "elderly_ratio",
    "day_night_ratio",
    "biz_per_10k",
    "food_accom_ratio",
    "illegal_per_10k"
]

reg_data = df_model[model_cols].dropna().copy()

z_cols = [
    "log_total_pop",
    "log_living_pop",
    "elderly_ratio",
    "day_night_ratio",
    "biz_per_10k",
    "food_accom_ratio",
    "illegal_per_10k"
]

for col in z_cols:
    reg_data[col + "_z"] = (reg_data[col] - reg_data[col].mean()) / reg_data[col].std()

print(reg_data.shape)
print(reg_data["cluster_type"].value_counts())

(100, 19)
cluster_type
주거형      32
상업형      24
혼합형      24
유동집중형    20
Name: count, dtype: int64


### 3-1. 범주형 변수 기준 수준 설정

`cluster_type`의 기준 범주(reference)를 **주거형**으로 설정한다.  
이후 회귀 계수는 "주거형 대비 해당 유형의 차이"로 해석된다.

In [7]:
# 기준모형을 주거형으로
reg_data["cluster_type"] = reg_data["cluster_type"].astype("category")

cats = list(reg_data["cluster_type"].cat.categories)
print(cats)

if "주거형" in cats:
    new_order = ["주거형"] + [c for c in cats if c != "주거형"]
    reg_data["cluster_type"] = reg_data["cluster_type"].cat.reorder_categories(
        new_order,
        ordered=False
    )

print(reg_data["cluster_type"].cat.categories)

['상업형', '유동집중형', '주거형', '혼합형']
Index(['주거형', '상업형', '유동집중형', '혼합형'], dtype='object')


## 4. 모델 추정: 단계적 회귀

### M1: 기본 구조변수 모형

클러스터 정보 없이 **인구·사업체·생활인구 등 구조변수만**으로 폐기물 배출량을 설명한다.  
연도 고정효과(`C(year)`)를 포함하고, 자치구 클러스터 robust 표준오차를 사용한다.

$$\log(\text{waste}) = \beta_0 + \beta_1 \text{pop}_z + \beta_2 \text{living}_z + \cdots + \gamma_t \cdot \text{year}_t + \varepsilon$$

In [8]:
# 기본 민서 변수만 
m1_formula = """
log_waste_total ~
    log_total_pop_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
"""

m1 = smf.ols(m1_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.603
Model:                            OLS   Adj. R-squared:                  0.558
Method:                 Least Squares   F-statistic:                     14.98
Date:                Wed, 13 May 2026   Prob (F-statistic):           5.09e-08
Time:                        13:16:06   Log-Likelihood:                -1.1367
No. Observations:                 100   AIC:                             24.27
Df Residuals:                      89   BIC:                             52.93
Df Model:                          10                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             11.8058      0

### M2: 클러스터 변수 추가

M1에 `cluster_type` 더미변수를 추가한다.  
자치구 유형(상업형/유동집중형/혼합형)이 주거형 대비 **절편을 얼마나 이동**시키는지 확인한다.  
R² 변화량으로 클러스터 변수의 **추가 설명력**을 평가한다.

In [9]:
# 클러스터변수 추가 
m2_formula = """
log_waste_total ~
    log_total_pop_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
    + C(cluster_type)
"""

m2 = smf.ols(m2_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.684
Model:                            OLS   Adj. R-squared:                  0.637
Method:                 Least Squares   F-statistic:                     26.00
Date:                Wed, 13 May 2026   Prob (F-statistic):           5.54e-11
Time:                        13:16:06   Log-Likelihood:                 10.351
No. Observations:                 100   AIC:                             7.297
Df Residuals:                      86   BIC:                             43.77
Df Model:                          13                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### M3: 클러스터 × 생활인구 교호작용

생활인구(`log_living_pop_z`)와 클러스터 유형의 **교호작용항(interaction)**을 추가한다.  
이는 "유동인구가 폐기물에 미치는 영향이 자치구 유형에 따라 다르다"는 가설을 검정한다.

예: 상업형 자치구에서는 유동인구 1단위 증가 시 폐기물이 더 많이 증가할 수 있다.

In [10]:
#M3. 클러스터 × 생활인구 interaction
m3_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
"""

m3 = smf.ols(m3_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m3.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.740
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     25.19
Date:                Wed, 13 May 2026   Prob (F-statistic):           3.44e-11
Time:                        13:16:06   Log-Likelihood:                 19.969
No. Observations:                 100   AIC:                            -5.937
Df Residuals:                      83   BIC:                             38.35
Df Model:                          16                                         
Covariance Type:              cluster                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

### M4: 클러스터 × (생활인구 + 음식숙박업) 교호작용

M3에 `food_accom_ratio_z × cluster_type` 교호작용을 추가한다.  
음식·숙박업 비율이 높은 자치구에서 폐기물 배출 패턴이 유형별로 다른지 확인한다.

In [11]:
#M4. 클러스터 × 생활인구 + 음식숙박업 interaction
m4_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
    + food_accom_ratio_z * C(cluster_type)
"""

m4 = smf.ols(m4_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m4.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.793
Method:                 Least Squares   F-statistic:                     838.9
Date:                Wed, 13 May 2026   Prob (F-statistic):           2.56e-29
Time:                        13:16:06   Log-Likelihood:                 41.982
No. Observations:                 100   AIC:                            -43.96
Df Residuals:                      80   BIC:                             8.139
Df Model:                          19                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

## 5. 모델 비교

M1~M4의 R², Adjusted R², AIC, BIC를 비교하여 최적 모형을 선정한다.

- **R² / Adj R²**: 높을수록 설명력 우수
- **AIC / BIC**: 낮을수록 모형 적합도·간결성 우수 (BIC가 과적합에 더 엄격)

In [12]:
model_compare = pd.DataFrame({
    "model": [
        "M1_basic",
        "M2_add_cluster",
        "M3_living_interaction",
        "M4_living_food_interaction"
    ],
    "nobs": [m1.nobs, m2.nobs, m3.nobs, m4.nobs],
    "r2": [m1.rsquared, m2.rsquared, m3.rsquared, m4.rsquared],
    "adj_r2": [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj, m4.rsquared_adj],
    "aic": [m1.aic, m2.aic, m3.aic, m4.aic],
    "bic": [m1.bic, m2.bic, m3.bic, m4.bic]
})

model_compare

,model,nobs,r2,adj_r2,aic,bic
0,M1_basic,100.0,0.602948,0.558336,24.273383,52.930255
1,M2_add_cluster,100.0,0.684454,0.636755,7.297212,43.769595
2,M3_living_interaction,100.0,0.739667,0.689483,-5.937303,38.350591
3,M4_living_food_interaction,100.0,0.832382,0.792573,-43.964658,8.138746


## 6. 민감도 분석: 강서구 제외

강서구는 마곡산업단지로 인해 이상치(outlier)일 가능성이 있다.  
강서구를 제외하고 M4를 재추정하여, 모형의 **안정성(robustness)** 을 확인한다.

In [13]:
# 강서구 제외 분석
reg_no_gangseo = reg_data[reg_data["gu"] != "강서구"].copy()

m4_no_gangseo = smf.ols(m4_formula, data=reg_no_gangseo).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_no_gangseo["gu"]}
)

sensitivity = pd.DataFrame({
    "model": ["강서구 포함 모델", "강서구 없는 모델"],
    "nobs": [m4.nobs, m4_no_gangseo.nobs],
    "r2": [m4.rsquared, m4_no_gangseo.rsquared],
    "adj_r2": [m4.rsquared_adj, m4_no_gangseo.rsquared_adj],
    "aic": [m4.aic, m4_no_gangseo.aic],
    "bic": [m4.bic, m4_no_gangseo.bic]
})

sensitivity

,model,nobs,r2,adj_r2,aic,bic
0,강서구 포함 모델,100.0,0.832382,0.792573,-43.964658,8.138746
1,강서구 없는 모델,96.0,0.854946,0.818683,-108.988701,-57.701737


### 6-1. 계수 변화 비교

강서구 포함/제외 시 각 계수가 얼마나 변하는지 대조한다.  
특정 계수가 크게 흔들린다면 강서구가 해당 관계를 지배하고 있다는 뜻이다.

In [14]:
# interaction 계수 방향 보기
coef_compare = pd.DataFrame({
    "coef_all": m4.params,
    "p_all": m4.pvalues,
    "coef_no_gangseo": m4_no_gangseo.params,
    "p_no_gangseo": m4_no_gangseo.pvalues
})

coef_compare["coef_diff"] = coef_compare["coef_no_gangseo"] - coef_compare["coef_all"]

coef_compare

,coef_all,p_all,coef_no_gangseo,p_no_gangseo,coef_diff
Intercept,11.657426,0.000000,11.666774,0.000000e+00,0.009348
C(year)[T.2021],0.024495,0.715127,-0.027432,4.391202e-01,-0.051928
C(year)[T.2022],0.051532,0.420871,0.029852,5.790830e-01,-0.021680
C(year)[T.2023],0.044641,0.599445,-0.006180,9.250812e-01,-0.050821
C(cluster_type)[T.상업형],0.073175,0.396867,0.083366,2.407935e-01,0.010191
C(cluster_type)[T.유동집중형],0.053368,0.426084,0.109156,4.050217e-03,0.055788
C(cluster_type)[T.혼합형],0.351480,0.000095,0.124247,3.087796e-03,-0.227233
log_total_pop_z,0.211684,0.294421,0.519065,2.476618e-05,0.307380
elderly_ratio_z,-0.027277,0.572068,0.035860,1.513191e-01,0.063137
day_night_ratio_z,0.195405,0.226950,0.334077,6.423201e-03,0.138671


## 7. Nested F-test (모형 비교 검정)

M1⊂M2⊂M3⊂M4의 중첩 관계를 이용해 **F-검정**으로 변수 추가의 통계적 유의성을 검정한다.  
robust SE 하에서는 F-test가 정확하지 않으므로, 일반 OLS로 재추정하여 검정한다.

> H₀: 추가 변수의 계수가 모두 0 (= 추가 변수가 불필요)

In [15]:
# robust covariance가 아닌 일반 OLS 기준 nested F-test용
m1_plain = smf.ols(m1_formula, data=reg_data).fit()
m2_plain = smf.ols(m2_formula, data=reg_data).fit()
m3_plain = smf.ols(m3_formula, data=reg_data).fit()
m4_plain = smf.ols(m4_formula, data=reg_data).fit()

print("M1 vs M2:", m2_plain.compare_f_test(m1_plain))
print("M2 vs M3:", m3_plain.compare_f_test(m2_plain))
print("M3 vs M4:", m4_plain.compare_f_test(m3_plain))

M1 vs M2: (7.404603635575735, 0.00018046329134539815, 3.0)
M2 vs M3: (5.867790058943603, 0.0011029375872370912, 3.0)
M3 vs M4: (14.75018724486331, 9.86913614363026e-08, 3.0)


## 8. 잔차 분석: 과다/과소 배출 자치구 식별

**강서구 제외 데이터로 추정한 M4**(기준 모형)를 전체 자치구에 적용하여 **기대 배출량**을 예측한다.  
실제 배출량과 예측값의 차이(잔차)를 통해 **구조적으로 기대보다 많이/적게 배출하는 자치구**를 식별한다.

- `residual_pct > 0`: 기대 대비 **초과 배출** (정책 개입 필요)
- `residual_pct < 0`: 기대 대비 **과소 배출** (모범 사례 후보)

In [16]:
# 1. 강서구 제외 데이터로 기준모형 적합
baseline_data = reg_data[reg_data["gu"] != "강서구"].copy()

baseline_model = smf.ols(m4_formula, data=baseline_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": baseline_data["gu"]}
)

print(baseline_model.summary())
# 2. 전체 데이터에 대해 예측
reg_data_pred = reg_data.copy()

reg_data_pred["pred_log_waste"] = baseline_model.predict(reg_data_pred)

# 로그값을 원래 폐기물 총량 단위로 변환
reg_data_pred["pred_waste_total"] = np.exp(reg_data_pred["pred_log_waste"])

# 잔차
reg_data_pred["residual"] = reg_data_pred["waste_total"] - reg_data_pred["pred_waste_total"]

# 로그 잔차: 비율 해석에 좋음
reg_data_pred["log_residual"] = (
    reg_data_pred["log_waste_total"] - reg_data_pred["pred_log_waste"]
)

# 실제가 기대보다 몇 % 높은지
reg_data_pred["residual_pct"] = (np.exp(reg_data_pred["log_residual"]) - 1) * 100

reg_data_pred[[
    "year", "gu", "cluster_type",
    "waste_total", "pred_waste_total",
    "residual", "residual_pct"
]].sort_values("residual_pct", ascending=False).head(15)

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.855
Model:                            OLS   Adj. R-squared:                  0.819
Method:                 Least Squares   F-statistic:                     828.0
Date:                Wed, 13 May 2026   Prob (F-statistic):           3.62e-28
Time:                        13:16:06   Log-Likelihood:                 74.494
No. Observations:                  96   AIC:                            -109.0
Df Residuals:                      76   BIC:                            -57.70
Df Model:                          19                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,year,gu,cluster_type,waste_total,pred_waste_total,residual,residual_pct
28,2021,강서구,혼합형,751795,186118.677824,565676.322176,303.933130
53,2022,강서구,혼합형,585756,194532.720105,391223.279895,201.109243
78,2023,강서구,혼합형,490315,189532.915866,300782.084134,158.696490
70,2022,용산구,상업형,216363,141552.045035,74810.954965,52.850494
95,2023,용산구,상업형,208742,141067.234686,67674.765314,47.973412
37,2021,마포구,혼합형,199008,146534.207689,52473.792311,35.809927
10,2020,동대문구,유동집중형,161082,124188.102082,36893.897918,29.708078
81,2023,구로구,혼합형,171610,146484.403950,25125.596050,17.152404
67,2022,송파구,혼합형,301830,264173.162372,37656.837628,14.254604
54,2022,관악구,주거형,161574,141504.835878,20069.164122,14.182670


### 8-1. 2023년 자치구별 잔차 랭킹

In [17]:
resid_2023 = reg_data_pred[reg_data_pred["year"] == 2023].copy()

resid_2023[[
    "gu", "cluster_type",
    "waste_total", "pred_waste_total",
    "residual", "residual_pct"
]].sort_values("residual_pct", ascending=False)

,gu,cluster_type,waste_total,pred_waste_total,residual,residual_pct
78,강서구,혼합형,490315,189532.915866,300782.084134,158.696490
95,용산구,상업형,208742,141067.234686,67674.765314,47.973412
81,구로구,혼합형,171610,146484.403950,25125.596050,17.152404
79,관악구,주거형,146875,134653.402259,12221.597741,9.076338
82,금천구,혼합형,101244,93330.191137,7913.808863,8.479366
84,도봉구,주거형,102431,94439.922858,7991.077142,8.461546
97,종로구,상업형,97419,91636.360281,5782.639719,6.310421
77,강북구,유동집중형,88886,86810.301421,2075.698579,2.391074
92,송파구,혼합형,262182,259877.101937,2304.898063,0.886918
94,영등포구,상업형,166688,167798.716831,-1110.716831,-0.661934


잔차 분석 결과와 모델 비교표를 CSV로 내보낸다.

In [18]:
# Cell 추가
resid_2023[["gu","cluster_type","waste_total","pred_waste_total","residual","residual_pct"]].to_csv("resid_by_gu_2023.csv", index=False)
model_compare.to_csv("model_compare.csv", index=False)

## 9. 강서구 전용 회귀모형

강서구는 24개 구 모형에서 일관된 이상치로 나타난다.  
강서구 4개년 데이터만으로 **전용 소규모 모형**을 적합하여,  
"다른 모형 구조를 써도 강서구의 과다 배출 패턴이 재현되는지" 확인한다.

> ⚠️ n=4로 자유도가 극히 제한적이므로, 통계적 검정력보다 **방향성 확인** 목적이다.

In [19]:
# ════════════════════════════════════════════════════════════
# 강서구 전용 회귀모델 — 노트북에 Cell 추가해서 실행
# ════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# ── 1. 강서구만 추출 ──────────────────────────────────────
gs = reg_data[reg_data["gu"] == "강서구"].copy()
print(f"강서구 관측치: {len(gs)}행")
print(gs[["year", "log_waste_total", "log_living_pop_z", "biz_per_10k_z"]])

# ── 2. 강서구 전용 모델 (변수 2개) ────────────────────────
gs_formula = "log_waste_total ~ log_living_pop_z + biz_per_10k_z"
gs_model = smf.ols(gs_formula, data=gs).fit()
print("\n" + "="*60)
print("강서구 전용 모델 (n=4, df_resid=1)")
print("="*60)
print(gs_model.summary())

# ── 3. 변수 1개짜리도 비교용으로 ──────────────────────────
gs_model_1 = smf.ols("log_waste_total ~ log_living_pop_z", data=gs).fit()
gs_model_2 = smf.ols("log_waste_total ~ biz_per_10k_z", data=gs).fit()

print("\n── 모델 비교 ──")
print(f"생활인구만:        R²={gs_model_1.rsquared:.4f}, AIC={gs_model_1.aic:.2f}")
print(f"사업체밀도만:      R²={gs_model_2.rsquared:.4f}, AIC={gs_model_2.aic:.2f}")
print(f"생활인구+사업체:   R²={gs_model.rsquared:.4f}, AIC={gs_model.aic:.2f}")

# ── 4. 세 가지 예측값 비교 ────────────────────────────────
gs["pred_main_log"] = baseline_model.predict(gs)          # 본모델 (24개구)
gs["pred_gs_log"] = gs_model.predict(gs)                   # 강서구 전용

gs["actual_waste"] = np.exp(gs["log_waste_total"])
gs["pred_main_waste"] = np.exp(gs["pred_main_log"])
gs["pred_gs_waste"] = np.exp(gs["pred_gs_log"])

gs["resid_main"] = gs["actual_waste"] - gs["pred_main_waste"]
gs["resid_gs"] = gs["actual_waste"] - gs["pred_gs_waste"]
gs["resid_main_pct"] = gs["resid_main"] / gs["pred_main_waste"] * 100
gs["resid_gs_pct"] = gs["resid_gs"] / gs["pred_gs_waste"] * 100

print("\n── 강서구 연도별 비교 ──")
result_cols = ["year", "actual_waste", "pred_main_waste", "pred_gs_waste",
               "resid_main_pct", "resid_gs_pct"]
print(gs[result_cols].to_string(index=False, float_format="%.0f"))

# ── 5. CSV 내보내기 ───────────────────────────────────────
# 5-1. 강서구 전용모델 결과
gs[result_cols].to_csv("gangseo_model_result.csv", index=False)

# 5-2. 강서구 전용모델 계수
gs_coef = pd.DataFrame({
    "term": gs_model.params.index,
    "coef": gs_model.params.values,
    "pvalue": gs_model.pvalues.values
})
gs_coef.to_csv("gangseo_model_coef.csv", index=False)

print("\n✓ gangseo_model_result.csv 저장 완료")
print("✓ gangseo_model_coef.csv 저장 완료")

# ── 6. 대시보드용 요약 출력 ───────────────────────────────
print("\n" + "="*60)
print("📊 대시보드 서사 요약 (2023년 기준)")
print("="*60)
g23 = gs[gs["year"] == 2023].iloc[0]
print(f"  실제 배출량:           {g23['actual_waste']:>12,.0f} 톤")
print(f"  본모델 예측 (24개구):  {g23['pred_main_waste']:>12,.0f} 톤  → +{g23['resid_main_pct']:.0f}% 초과")
print(f"  강서구 전용모델 예측:  {g23['pred_gs_waste']:>12,.0f} 톤  → +{g23['resid_gs_pct']:.0f}% 초과")
print(f"\n  → 어떤 모델로 봐도 강서구는 비정상 과다배출")

강서구 관측치: 4행
    year  log_waste_total  log_living_pop_z  biz_per_10k_z
3   2020        12.169393          1.015837      -0.399705
28  2021        13.530219          0.996260      -0.388547
53  2022        13.280659          0.992982      -0.364118
78  2023        13.102803          0.968675      -0.347735

강서구 전용 모델 (n=4, df_resid=1)
                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.374
Model:                            OLS   Adj. R-squared:                 -0.878
Method:                 Least Squares   F-statistic:                    0.2989
Date:                Wed, 13 May 2026   Prob (F-statistic):              0.791
Time:                        13:16:06   Log-Likelihood:                -2.0800
No. Observations:                   4   AIC:                             10.16
Df Residuals:                       1   BIC:                             8.319
Df Model:                       

C:\Users\User\anaconda3\Lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 4 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


## 10. 강서구 더미 모형 (M5, M6)

전체 데이터에 `is_gangseo` 더미변수를 추가하여, 강서구의 **체계적 이탈**을 모형 내에서 흡수한다.

| 모형 | 추가 내용 | 의미 |
|------|-----------|------|
| **M5** | M4 + `is_gangseo` 더미 | 강서구의 **절편 차이**만 허용 |
| **M6** | M5 + `is_gangseo × 생활인구`, `is_gangseo × 사업체밀도` | 강서구의 **기울기 차이**까지 허용 |

M6에서 교호작용이 유의하면, 강서구는 단순히 수준만 다른 게 아니라 **변수 간 관계 구조 자체**가 다른 것이다.

In [20]:
# ════════════════════════════════════════════════════════════
# [Cell A] 강서구 더미변수 생성 + M5, M6 모델
# ════════════════════════════════════════════════════════════
 
# 더미변수 생성
reg_data["is_gangseo"] = (reg_data["gu"] == "강서구").astype(int)
 
# ── M5: M4 + 강서구 더미 (절편 차이만) ──
m5_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
    + food_accom_ratio_z * C(cluster_type)
    + is_gangseo
"""
 
m5 = smf.ols(m5_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)
 
# ── M6: M5 + 강서구 교호작용 (기울기 차이까지) ──
m6_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
    + food_accom_ratio_z * C(cluster_type)
    + is_gangseo
    + is_gangseo : log_living_pop_z
    + is_gangseo : biz_per_10k_z
"""
 
m6 = smf.ols(m6_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)
 
print("=" * 60)
print("M5: M4 + 강서구 더미")
print("=" * 60)
print(m5.summary())
 
print("\n" + "=" * 60)
print("M6: M5 + 강서구 교호작용")
print("=" * 60)
print(m6.summary())
 
 

M5: M4 + 강서구 더미
                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.853
Model:                            OLS   Adj. R-squared:                  0.815
Method:                 Least Squares   F-statistic:                     7294.
Date:                Wed, 13 May 2026   Prob (F-statistic):           1.15e-40
Time:                        13:16:07   Log-Likelihood:                 48.452
No. Observations:                 100   AIC:                            -54.90
Df Residuals:                      79   BIC:                           -0.1950
Df Model:                          20                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

C:\Users\User\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 22, but rank is 20
  warnings.warn('covariance of constraints does not have full '


### 10-1. M1~M6 전체 모델 비교

6개 모형의 적합도를 종합 비교하고, 강서구 관련 계수의 크기와 유의성을 확인한다.

In [21]:
# ════════════════════════════════════════════════════════════
# [Cell B] 모델 비교표
# ════════════════════════════════════════════════════════════
 
model_compare_v2 = pd.DataFrame({
    "model": [
        "M1_basic",
        "M2_add_cluster",
        "M3_living_interaction",
        "M4_living_food_interaction",
        "M5_gangseo_dummy",
        "M6_gangseo_interaction"
    ],
    "nobs": [m1.nobs, m2.nobs, m3.nobs, m4.nobs, m5.nobs, m6.nobs],
    "r2": [m1.rsquared, m2.rsquared, m3.rsquared, m4.rsquared, m5.rsquared, m6.rsquared],
    "adj_r2": [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj, m4.rsquared_adj, m5.rsquared_adj, m6.rsquared_adj],
    "aic": [m1.aic, m2.aic, m3.aic, m4.aic, m5.aic, m6.aic],
    "bic": [m1.bic, m2.bic, m3.bic, m4.bic, m5.bic, m6.bic]
})
 
print("\n── M1 → M6 모델 비교 ──")
print(model_compare_v2.to_string(index=False))
 
# 강서구 더미 계수만 뽑기
print("\n── 강서구 관련 계수 ──")
for name, model in [("M5", m5), ("M6", m6)]:
    gs_terms = [t for t in model.params.index if "gangseo" in t.lower()]
    for t in gs_terms:
        print(f"  {name} | {t}: coef={model.params[t]:.4f}, p={model.pvalues[t]:.4f}")
 
 


── M1 → M6 모델 비교 ──
                     model  nobs       r2   adj_r2        aic        bic
                  M1_basic 100.0 0.602948 0.558336  24.273383  52.930255
            M2_add_cluster 100.0 0.684454 0.636755   7.297212  43.769595
     M3_living_interaction 100.0 0.739667 0.689483  -5.937303  38.350591
M4_living_food_interaction 100.0 0.832382 0.792573 -43.964658   8.138746
          M5_gangseo_dummy 100.0 0.852726 0.815441 -54.903580  -0.195006
    M6_gangseo_interaction 100.0 0.878636 0.843961 -70.254185 -10.335271

── 강서구 관련 계수 ──
  M5 | is_gangseo: coef=0.6614, p=0.0133
  M6 | is_gangseo: coef=31.2812, p=0.0000
  M6 | is_gangseo:log_living_pop_z: coef=-37.3165, p=0.0000
  M6 | is_gangseo:biz_per_10k_z: coef=-17.3975, p=0.0000


### 10-2. 최종 모형 기준 잔차 분석

M5/M6 중 Adj R²가 더 높은 모형을 선택하여 최종 잔차를 산출한다.  
강서구 더미를 포함했을 때 다른 자치구의 잔차 순위가 어떻게 변하는지 확인한다.

In [22]:
# ════════════════════════════════════════════════════════════
# [Cell C] 최종 모델로 잔차 분석 + CSV 내보내기
# ════════════════════════════════════════════════════════════
 
# M5 or M6 중 더 나은 모델 선택 (여기선 둘 다 내보냄)
best = m6 if m6.rsquared_adj > m5.rsquared_adj else m5
best_name = "M6" if m6.rsquared_adj > m5.rsquared_adj else "M5"
print(f"\n선택된 모델: {best_name} (adj_R²={best.rsquared_adj:.4f})")
 
# 예측 + 잔차
reg_data_v2 = reg_data.copy()
reg_data_v2["pred_log_waste"] = best.predict(reg_data_v2)
reg_data_v2["pred_waste_total"] = np.exp(reg_data_v2["pred_log_waste"])
reg_data_v2["residual"] = reg_data_v2["waste_total"] - reg_data_v2["pred_waste_total"]
reg_data_v2["residual_pct"] = reg_data_v2["residual"] / reg_data_v2["pred_waste_total"] * 100
 
# 2023년 잔차
resid_2023_v2 = reg_data_v2[reg_data_v2["year"] == 2023].copy()
 
print(f"\n── 2023년 잔차 (상위 10, {best_name} 기준) ──")
print(resid_2023_v2[["gu", "cluster_type", "is_gangseo",
                      "waste_total", "pred_waste_total",
                      "residual", "residual_pct"
]].sort_values("residual_pct", ascending=False).head(10).to_string(index=False))
 
# ── CSV 내보내기 ──
 
# 1. 모델 비교표 (M1~M6)
model_compare_v2.to_csv("model_compare_v2.csv", index=False)
 
# 2. 최종 모델 계수
coef_best = pd.DataFrame({
    "term": best.params.index,
    "coef": best.params.values,
    "pvalue": best.pvalues.values
})
coef_best.to_csv("best_model_coef.csv", index=False)
 
# 3. 2023 잔차 (전 자치구)
resid_2023_v2[["gu", "cluster_type", "is_gangseo",
               "waste_total", "pred_waste_total",
               "residual", "residual_pct"
]].to_csv("resid_by_gu_2023_v2.csv", index=False)
 
# 4. 전체 연도 잔차 (강서구 연도별 추이 시각화용)
reg_data_v2[["year", "gu", "cluster_type", "is_gangseo",
             "waste_total", "pred_waste_total",
             "residual", "residual_pct"
]].to_csv("resid_all_years.csv", index=False)
 
# 5. 강서구 포함 vs 제외 sensitivity
sensitivity_v2 = pd.DataFrame({
    "model": ["M4 (강서구 제외 학습)", f"{best_name} (강서구 더미 포함)"],
    "r2": [m4_no_gangseo.rsquared, best.rsquared],
    "adj_r2": [m4_no_gangseo.rsquared_adj, best.rsquared_adj],
    "aic": [m4_no_gangseo.aic, best.aic]
})
sensitivity_v2.to_csv("sensitivity_v2.csv", index=False)
 
print("\n✓ model_compare_v2.csv")
print("✓ best_model_coef.csv")
print("✓ resid_by_gu_2023_v2.csv")
print("✓ resid_all_years.csv")
print("✓ sensitivity_v2.csv")
print("\n내보내기 완료!")


선택된 모델: M6 (adj_R²=0.8440)

── 2023년 잔차 (상위 10, M6 기준) ──
  gu cluster_type  is_gangseo  waste_total  pred_waste_total     residual  residual_pct
 용산구          상업형           0       208742     139330.122232 69411.877768     49.818285
 구로구          혼합형           0       171610     150032.486537 21577.513463     14.381894
 도봉구          주거형           0       102431      91778.985196 10652.014804     11.606159
 관악구          주거형           0       146875     132092.975738 14782.024262     11.190621
 금천구          혼합형           0       101244      92038.715443  9205.284557     10.001535
 종로구          상업형           0        97419      89558.586893  7860.413107      8.776839
 강북구        유동집중형           0        88886      85565.282521  3320.717479      3.880917
 동작구          주거형           0       116719     115551.020356  1167.979644      1.010791
영등포구          상업형           0       166688     165042.220574  1645.779426      0.997187
 송파구          혼합형           0       262182     260867.334024 

In [23]:
# ════════════════════════════════════════════════════════════
# [Cell C] 모든 모델 2023 잔차 비교 + CSV 내보내기
# ════════════════════════════════════════════════════════════

# 최종 모델 선택
best = m6 if m6.rsquared_adj > m5.rsquared_adj else m5
best_name = "M6" if m6.rsquared_adj > m5.rsquared_adj else "M5"
print(f"선택된 모델: {best_name} (adj_R²={best.rsquared_adj:.4f})\n")

# 각 모델 예측값 계산
compare = reg_data.copy()
compare["pred_m4"]          = np.exp(m4.predict(compare))
compare["pred_m4_no_gs"]    = np.exp(baseline_model.predict(compare))
compare["pred_best"]        = np.exp(best.predict(compare))

# 잔차율 계산
for tag, pred_col in [("m4", "pred_m4"), ("m4_no_gs", "pred_m4_no_gs"), ("best", "pred_best")]:
    compare[f"resid_{tag}"]     = compare["waste_total"] - compare[pred_col]
    compare[f"resid_pct_{tag}"] = compare[f"resid_{tag}"] / compare[pred_col] * 100

# 2023년만
c23 = compare[compare["year"] == 2023].copy()

print("── 2023 잔차율(%) 비교: M4 vs M4(강서구제외) vs " + best_name + " ──")
print(c23[["gu", "cluster_type", "is_gangseo", "waste_total",
           "pred_m4", "resid_pct_m4",
           "pred_m4_no_gs", "resid_pct_m4_no_gs",
           "pred_best", "resid_pct_best"
]].sort_values("resid_pct_m4_no_gs", ascending=False).to_string(index=False, float_format="%.1f"))

# 강서구만 하이라이트
print("\n── 강서구 연도별 3모델 비교 ──")
gs_all = compare[compare["gu"] == "강서구"][
    ["year", "waste_total", "pred_m4", "resid_pct_m4",
     "pred_m4_no_gs", "resid_pct_m4_no_gs",
     "pred_best", "resid_pct_best"]
]
print(gs_all.to_string(index=False, float_format="%.1f"))

# ── CSV 내보내기 ──

# 1. 모델 비교표
model_compare_v2.to_csv("model_compare_v2.csv", index=False)

# 2. 최종 모델 계수
coef_best = pd.DataFrame({
    "term": best.params.index,
    "coef": best.params.values,
    "pvalue": best.pvalues.values
})
coef_best.to_csv("best_model_coef.csv", index=False)

# 3. 2023 잔차 (3모델 비교)
c23[["gu", "cluster_type", "is_gangseo", "waste_total",
     "pred_m4", "resid_pct_m4",
     "pred_m4_no_gs", "resid_pct_m4_no_gs",
     "pred_best", "resid_pct_best"
]].to_csv("resid_2023_all_models.csv", index=False, encoding="utf-8-sig"
)

# 4. 전체 연도 (강서구 추이 차트용)
compare[["year", "gu", "cluster_type", "is_gangseo", "waste_total",
         "pred_m4", "pred_m4_no_gs", "pred_best",
         "resid_pct_m4", "resid_pct_m4_no_gs", "resid_pct_best"
]].to_csv("resid_all_years.csv", index=False)

# 5. sensitivity
sensitivity_v2 = pd.DataFrame({
    "model": ["M4 (전체)", "M4 (강서구 제외 학습)", f"{best_name} (강서구 더미)"],
    "r2": [m4.rsquared, m4_no_gangseo.rsquared, best.rsquared],
    "adj_r2": [m4.rsquared_adj, m4_no_gangseo.rsquared_adj, best.rsquared_adj],
    "aic": [m4.aic, m4_no_gangseo.aic, best.aic]
})
sensitivity_v2.to_csv("sensitivity_v2.csv", index=False)

print("\n✓ model_compare_v2.csv")
print("✓ best_model_coef.csv")
print("✓ resid_2023_all_models.csv")
print("✓ resid_all_years.csv")
print("✓ sensitivity_v2.csv")

선택된 모델: M6 (adj_R²=0.8440)

── 2023 잔차율(%) 비교: M4 vs M4(강서구제외) vs M6 ──
  gu cluster_type  is_gangseo  waste_total  pred_m4  resid_pct_m4  pred_m4_no_gs  resid_pct_m4_no_gs  pred_best  resid_pct_best
 강서구          혼합형           1       490315 410148.6          19.5       189532.9               158.7   694217.0           -29.4
 용산구          상업형           0       208742 142243.1          46.8       141067.2                48.0   139330.1            49.8
 구로구          혼합형           0       171610 190266.5          -9.8       146484.4                17.2   150032.5            14.4
 관악구          주거형           0       146875 134522.3           9.2       134653.4                 9.1   132093.0            11.2
 금천구          혼합형           0       101244  94402.8           7.2        93330.2                 8.5    92038.7            10.0
 도봉구          주거형           0       102431  89713.5          14.2        94439.9                 8.5    91779.0            11.6
 종로구          상업형           0   

In [24]:
# ============================================================
# 급증 전 데이터만 제외하고 재학습
# - 강서구: 2020년 제외
# - 용산구: 2020~2021년 제외
# 목적:
#   급증 이후의 현재 배출 구조를 기준으로 회귀모형 재학습
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. 급증 전 데이터 정의
pre_spike_mask = (
    ((reg_data["gu"] == "강서구") & (reg_data["year"].isin([2020]))) |
    ((reg_data["gu"] == "용산구") & (reg_data["year"].isin([2020, 2021])))
)

# 2. 학습 데이터 생성
reg_train_after_spike = reg_data.loc[~pre_spike_mask].copy()
reg_all_for_pred = reg_data.copy()

print("전체 데이터:", reg_data.shape)
print("제외 데이터 수:", pre_spike_mask.sum())

display(
    reg_data.loc[
        pre_spike_mask,
        ["year", "gu", "waste_total", "cluster_type"]
    ].sort_values(["gu", "year"])
)

print("재학습 데이터:", reg_train_after_spike.shape)

# 3. 급증 전 데이터 제외 후 M4 재학습
m4_after_spike = smf.ols(
    m4_formula,
    data=reg_train_after_spike
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_train_after_spike["gu"]}
)

print(m4_after_spike.summary())

전체 데이터: (100, 20)
제외 데이터 수: 3


,year,gu,waste_total,cluster_type
3,2020,강서구,192797,혼합형
20,2020,용산구,110255,상업형
45,2021,용산구,107173,상업형


재학습 데이터: (97, 20)
                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.880
Model:                            OLS   Adj. R-squared:                  0.851
Method:                 Least Squares   F-statistic:                     73.56
Date:                Wed, 13 May 2026   Prob (F-statistic):           9.09e-17
Time:                        13:16:07   Log-Likelihood:                 56.292
No. Observations:                  97   AIC:                            -72.58
Df Residuals:                      77   BIC:                            -21.09
Df Model:                          19                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------

In [25]:
# ============================================================
# 급증 전 데이터 제외 학습모형을 전체 데이터에 적용해 잔차 계산
# ============================================================

reg_pred_after_spike = reg_all_for_pred.copy()

# 1. 전체 데이터 예측
reg_pred_after_spike["pred_log_waste_after_spike"] = m4_after_spike.predict(reg_pred_after_spike)
reg_pred_after_spike["pred_waste_after_spike"] = np.exp(
    reg_pred_after_spike["pred_log_waste_after_spike"]
)

# 2. 실제 - 예측
reg_pred_after_spike["residual_after_spike"] = (
    reg_pred_after_spike["waste_total"] - reg_pred_after_spike["pred_waste_after_spike"]
)

# 3. 비율 잔차
reg_pred_after_spike["log_residual_after_spike"] = (
    reg_pred_after_spike["log_waste_total"] - reg_pred_after_spike["pred_log_waste_after_spike"]
)

reg_pred_after_spike["residual_pct_after_spike"] = (
    np.exp(reg_pred_after_spike["log_residual_after_spike"]) - 1
) * 100

# 4. 2023년 결과만 확인
resid_2023_after_spike = reg_pred_after_spike[
    reg_pred_after_spike["year"] == 2023
].copy()

resid_2023_after_spike_out = resid_2023_after_spike[[
    "gu", "cluster_type",
    "waste_total",
    "pred_waste_after_spike",
    "residual_after_spike",
    "residual_pct_after_spike"
]].copy()

resid_2023_after_spike_out["pred_waste_after_spike"] = resid_2023_after_spike_out["pred_waste_after_spike"].round(0)
resid_2023_after_spike_out["residual_after_spike"] = resid_2023_after_spike_out["residual_after_spike"].round(0)
resid_2023_after_spike_out["residual_pct_after_spike"] = resid_2023_after_spike_out["residual_pct_after_spike"].round(1)

display(
    resid_2023_after_spike_out.sort_values(
        "residual_pct_after_spike",
        ascending=False
    )
)

,gu,cluster_type,waste_total,pred_waste_after_spike,residual_after_spike,residual_pct_after_spike
97,종로구,상업형,97419,83228.0,14191.0,17.1
84,도봉구,주거형,102431,88349.0,14082.0,15.9
95,용산구,상업형,208742,180491.0,28251.0,15.7
79,관악구,주거형,146875,131072.0,15803.0,12.1
82,금천구,혼합형,101244,93295.0,7949.0,8.5
78,강서구,혼합형,490315,457703.0,32612.0,7.1
96,은평구,주거형,123761,115674.0,8087.0,7.0
94,영등포구,상업형,166688,160006.0,6682.0,4.2
98,중구,상업형,132481,128340.0,4141.0,3.2
77,강북구,유동집중형,88886,86227.0,2659.0,3.1


In [26]:
# ============================================================
# 대시보드용 잔차 CSV 저장
# ============================================================

resid_export = resid_2023_after_spike[[
    "gu", "cluster_type",
    "waste_total",
    "pred_waste_after_spike",
    "residual_after_spike",
    "residual_pct_after_spike"
]].copy()

resid_export["is_gangseo"] = (resid_export["gu"] == "강서구").astype(int)
resid_export["is_yongsan"] = (resid_export["gu"] == "용산구").astype(int)

# R 코드가 우선 인식하는 컬럼명으로 맞추기
resid_export = resid_export.rename(columns={
    "pred_waste_after_spike": "pred_waste_baseline",
    "residual_after_spike": "baseline_residual",
    "residual_pct_after_spike": "baseline_residual_pct"
})

resid_export["pred_waste_baseline"] = resid_export["pred_waste_baseline"].round(0)
resid_export["baseline_residual"] = resid_export["baseline_residual"].round(0)
resid_export["baseline_residual_pct"] = resid_export["baseline_residual_pct"].round(1)

resid_export = resid_export[[
    "gu", "cluster_type",
    "is_gangseo", "is_yongsan",
    "waste_total",
    "pred_waste_baseline",
    "baseline_residual",
    "baseline_residual_pct"
]]

out_path = "C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/regresssion/resid_by_gu_2023_v2.csv"

resid_export.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)

display(
    resid_export.sort_values(
        "baseline_residual_pct",
        ascending=False
    )
)

저장 완료: C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/regresssion/resid_by_gu_2023_v2.csv


,gu,cluster_type,is_gangseo,is_yongsan,waste_total,pred_waste_baseline,baseline_residual,baseline_residual_pct
97,종로구,상업형,0,0,97419,83228.0,14191.0,17.1
84,도봉구,주거형,0,0,102431,88349.0,14082.0,15.9
95,용산구,상업형,0,1,208742,180491.0,28251.0,15.7
79,관악구,주거형,0,0,146875,131072.0,15803.0,12.1
82,금천구,혼합형,0,0,101244,93295.0,7949.0,8.5
78,강서구,혼합형,1,0,490315,457703.0,32612.0,7.1
96,은평구,주거형,0,0,123761,115674.0,8087.0,7.0
94,영등포구,상업형,0,0,166688,160006.0,6682.0,4.2
98,중구,상업형,0,0,132481,128340.0,4141.0,3.2
77,강북구,유동집중형,0,0,88886,86227.0,2659.0,3.1


### 최종 회귀모형 식

최종 모형은 강서구 2020년, 용산구 2020~2021년 데이터를 제외한 뒤 재학습한 M4 모형이다. 종속변수는 자치구별 총 폐기물 발생량의 로그값이며, 설명변수는 인구 규모, 생활인구, 고령인구 비율, 주야간 생활인구비, 사업체 밀도, 음식·숙박업 비중, 불법투기 발생 수준, 연도, 자치구 유형을 포함한다.

$$
\log(Waste_i) =
\beta_0
+ \beta_1 \log(Pop_i)_z
+ \beta_2 Elderly_i{}_z
+ \beta_3 DayNight_i{}_z
+ \beta_4 Biz_i{}_z
+ \beta_5 Illegal_i{}_z
+ \gamma_{year}
+ \delta_{cluster}
+ \beta_6 \log(LivingPop_i)_z
+ \beta_7 FoodAccom_i{}_z
+ \theta_1 \left[\log(LivingPop_i)_z \times Cluster_i\right]
+ \theta_2 \left[FoodAccom_i{}_z \times Cluster_i\right]
+ \epsilon_i
$$

이를 계수까지 포함하면 다음과 같다.

$$
\begin{aligned}
\widehat{\log(Waste_i)} =
& 11.6269 \\
& -0.0040 \cdot I(Year=2021)
+0.0019 \cdot I(Year=2022)
-0.0026 \cdot I(Year=2023) \\
& +0.0765 \cdot I(Cluster=상업형)
+0.0905 \cdot I(Cluster=유동집중형)
+0.4289 \cdot I(Cluster=혼합형) \\
& +0.2892 \cdot \log(Pop_i)_z
-0.0292 \cdot Elderly_i{}_z
+0.2529 \cdot DayNight_i{}_z \\
& -0.1436 \cdot Biz_i{}_z
-0.1377 \cdot Illegal_i{}_z \\
& +0.0248 \cdot \log(LivingPop_i)_z \\
& -0.1836 \cdot \log(LivingPop_i)_z \cdot I(Cluster=상업형) \\
& -0.0277 \cdot \log(LivingPop_i)_z \cdot I(Cluster=유동집중형) \\
& -0.0594 \cdot \log(LivingPop_i)_z \cdot I(Cluster=혼합형) \\
& +0.0158 \cdot FoodAccom_i{}_z \\
& -0.4087 \cdot FoodAccom_i{}_z \cdot I(Cluster=상업형) \\
& -0.0347 \cdot FoodAccom_i{}_z \cdot I(Cluster=유동집중형) \\
& +0.5281 \cdot FoodAccom_i{}_z \cdot I(Cluster=혼합형)
\end{aligned}
$$

여기서 기준 범주는 `2020년`, `주거형`이다.  
따라서 연도 더미와 군집 더미의 계수는 각각 2020년 및 주거형 대비 차이를 의미한다.  
또한 `_z`가 붙은 변수는 표준화된 변수이며, 각 계수는 해당 변수가 1표준편차 증가할 때 로그 폐기물 발생량이 얼마나 변하는지를 나타낸다.

In [27]:
# ============================================================
# 군집별 절편이 직접 보이도록 수정한 모델
# - cluster_type은 baseline 없이 4개 모두 출력
# - year는 2020년을 기준으로 처리
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 기존과 동일하게 급증 전 관측치 제외
pre_spike_mask = (
    ((reg_data["gu"] == "강서구") & (reg_data["year"].isin([2020]))) |
    ((reg_data["gu"] == "용산구") & (reg_data["year"].isin([2020, 2021])))
)

reg_story_all = reg_data.copy()
reg_story_train = reg_story_all.loc[~pre_spike_mask].copy()

# 군집 순서 명시
cluster_order = ["주거형", "상업형", "유동집중형", "혼합형"]

reg_story_train["cluster_type"] = pd.Categorical(
    reg_story_train["cluster_type"],
    categories=cluster_order,
    ordered=False
)

reg_story_all["cluster_type"] = pd.Categorical(
    reg_story_all["cluster_type"],
    categories=cluster_order,
    ordered=False
)

# interaction 변수
interaction_vars = [
    "log_living_pop_z",
    "elderly_ratio_z",
    "day_night_ratio_z",
    "food_accom_ratio_z"
]

# 기본 통제변수
base_controls = [
    "log_total_pop_z",
    "biz_per_10k_z",
    "illegal_per_10k_z"
]

# 모델식 생성
formula_parts = []
formula_parts.append("log_waste_total ~")
formula_parts.append("    0 + C(cluster_type)")
formula_parts.append("    + C(year, Treatment(reference=2020))")

for v in base_controls:
    formula_parts.append(f"    + {v}")

for v in interaction_vars:
    formula_parts.append(f"    + C(cluster_type):{v}")

formula_cluster_direct_v2 = "\n".join(formula_parts)

print("=" * 80)
print("[수정 모델식]")
print("=" * 80)
print(formula_cluster_direct_v2)

# 모델 적합
cluster_direct_model_v2 = smf.ols(
    formula_cluster_direct_v2,
    data=reg_story_train
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_story_train["gu"]}
)

print("\n" + "=" * 80)
print("[수정 모델 결과]")
print("=" * 80)
print(cluster_direct_model_v2.summary())

[수정 모델식]
log_waste_total ~
    0 + C(cluster_type)
    + C(year, Treatment(reference=2020))
    + log_total_pop_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + C(cluster_type):log_living_pop_z
    + C(cluster_type):elderly_ratio_z
    + C(cluster_type):day_night_ratio_z
    + C(cluster_type):food_accom_ratio_z

[수정 모델 결과]
                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.934
Model:                            OLS   Adj. R-squared:                  0.911
Method:                 Least Squares   F-statistic:                       nan
Date:                Wed, 13 May 2026   Prob (F-statistic):                nan
Time:                        13:16:07   Log-Likelihood:                 85.092
No. Observations:                  97   AIC:                            -118.2
Df Residuals:                      71   BIC:                            -51.24
Df Model:                          25   

In [28]:
# ============================================================
# 수정 모델 계수표 정리
# ============================================================

coef_cluster_direct_v2 = pd.DataFrame({
    "term": cluster_direct_model_v2.params.index,
    "coef": cluster_direct_model_v2.params.values,
    "pvalue": cluster_direct_model_v2.pvalues.values
})

coef_cluster_direct_v2["effect_pct_per_1sd"] = (
    np.exp(coef_cluster_direct_v2["coef"]) - 1
) * 100

coef_cluster_direct_v2["significant_5pct"] = coef_cluster_direct_v2["pvalue"] < 0.05
coef_cluster_direct_v2["significant_10pct"] = coef_cluster_direct_v2["pvalue"] < 0.10
coef_cluster_direct_v2["direction"] = np.where(coef_cluster_direct_v2["coef"] > 0, "+", "-")

print("=" * 80)
print("[수정 모델 전체 계수표]")
print("=" * 80)
display(coef_cluster_direct_v2)


# 군집별 절편만 보기
cluster_main_v2 = coef_cluster_direct_v2[
    coef_cluster_direct_v2["term"].str.contains(r"^C\(cluster_type\)\[", regex=True)
    & ~coef_cluster_direct_v2["term"].str.contains(":", regex=False)
].copy()

print("\n" + "=" * 80)
print("[군집별 절편]")
print("=" * 80)
display(cluster_main_v2)


# 군집별 interaction만 보기
cluster_interaction_v2 = coef_cluster_direct_v2[
    coef_cluster_direct_v2["term"].str.contains(r"C\(cluster_type\)\[.*\]:", regex=True)
].copy()

print("\n" + "=" * 80)
print("[군집별 interaction]")
print("=" * 80)
display(
    cluster_interaction_v2.sort_values(
        ["significant_5pct", "pvalue"],
        ascending=[False, True]
    )
)

[수정 모델 전체 계수표]


,term,coef,pvalue,effect_pct_per_1sd,significant_5pct,significant_10pct,direction
0,C(cluster_type)[주거형],11.518283,0.000000e+00,1.005362e+07,True,True,+
1,C(cluster_type)[상업형],11.841335,0.000000e+00,1.388748e+07,True,True,+
2,C(cluster_type)[유동집중형],11.920685,0.000000e+00,1.503436e+07,True,True,+
3,C(cluster_type)[혼합형],12.153588,0.000000e+00,1.897727e+07,True,True,+
4,"C(year, Treatment(reference=2020))[T.2021]",-0.026345,4.879289e-01,-2.600122e+00,False,False,-
5,"C(year, Treatment(reference=2020))[T.2022]",-0.037642,4.560522e-01,-3.694260e+00,False,False,-
6,"C(year, Treatment(reference=2020))[T.2023]",-0.026918,6.577481e-01,-2.655891e+00,False,False,-
7,log_total_pop_z,0.159335,1.679854e-01,1.727309e+01,False,False,+
8,biz_per_10k_z,-0.079711,3.704483e-01,-7.661668e+00,False,False,-
9,illegal_per_10k_z,-0.144708,1.204739e-07,-1.347247e+01,True,True,-



[군집별 절편]


,term,coef,pvalue,effect_pct_per_1sd,significant_5pct,significant_10pct,direction
0,C(cluster_type)[주거형],11.518283,0.0,1.005362e+07,True,True,+
1,C(cluster_type)[상업형],11.841335,0.0,1.388748e+07,True,True,+
2,C(cluster_type)[유동집중형],11.920685,0.0,1.503436e+07,True,True,+
3,C(cluster_type)[혼합형],12.153588,0.0,1.897727e+07,True,True,+



[군집별 interaction]


,term,coef,pvalue,effect_pct_per_1sd,significant_5pct,significant_10pct,direction
25,C(cluster_type)[혼합형]:food_accom_ratio_z,1.062456,4.217047e-48,189.346795,True,True,+
21,C(cluster_type)[혼합형]:day_night_ratio_z,1.099767,2.492715e-05,200.346474,True,True,+
20,C(cluster_type)[유동집중형]:day_night_ratio_z,0.498137,7.858962e-03,64.565219,True,True,+
17,C(cluster_type)[혼합형]:elderly_ratio_z,-0.252563,3.422284e-02,-22.319236,True,True,-
23,C(cluster_type)[상업형]:food_accom_ratio_z,-0.428219,3.870371e-02,-34.833114,True,True,-
12,C(cluster_type)[유동집중형]:log_living_pop_z,0.135296,4.571497e-02,14.487505,True,True,+
15,C(cluster_type)[상업형]:elderly_ratio_z,0.113107,4.622087e-02,11.975215,True,True,+
10,C(cluster_type)[주거형]:log_living_pop_z,0.096176,2.426173e-01,10.095310,False,False,+
14,C(cluster_type)[주거형]:elderly_ratio_z,-0.015293,5.529401e-01,-1.517701,False,False,-
18,C(cluster_type)[주거형]:day_night_ratio_z,-0.078143,5.906851e-01,-7.516809,False,False,-


# 최종 모델

In [32]:
# ============================================================
# PPT용 모델 성능 비교표 생성
# Model 1: 클러스터 없는 기본모형
# Model 2: 클러스터만 추가한 모형
# Model 3: 클러스터 + 상호작용항 최종모형
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# 1) 최종모델과 동일한 학습 데이터 구성
#    - 강서구 2020년 제외
#    - 용산구 2020~2021년 제외
# ------------------------------------------------------------

pre_spike_mask = (
    ((reg_data["gu"] == "강서구") & (reg_data["year"].isin([2020]))) |
    ((reg_data["gu"] == "용산구") & (reg_data["year"].isin([2020, 2021])))
)

train_df = reg_data.loc[~pre_spike_mask].copy()

# ------------------------------------------------------------
# 2) 군집 기준 순서 고정
#    주거형을 기준군집으로 설정
# ------------------------------------------------------------

cluster_order = ["주거형", "상업형", "유동집중형", "혼합형"]

train_df["cluster_type"] = pd.Categorical(
    train_df["cluster_type"],
    categories=cluster_order,
    ordered=False
)

# ------------------------------------------------------------
# 3) 모델식 정의
# ------------------------------------------------------------

# Model 1. 클러스터 없는 기본모형
formula_m1 = """
log_waste_total ~
    C(year, Treatment(reference=2020))
    + log_total_pop_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + food_accom_ratio_z
"""

# Model 2. 클러스터만 추가한 모형
formula_m2 = """
log_waste_total ~
    C(year, Treatment(reference=2020))
    + log_total_pop_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + food_accom_ratio_z
    + C(cluster_type, Treatment(reference="주거형"))
"""

# Model 3. 클러스터 + 상호작용항 최종모형
formula_m3 = """
log_waste_total ~
    C(year, Treatment(reference=2020))
    + log_total_pop_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + log_living_pop_z * C(cluster_type, Treatment(reference="주거형"))
    + elderly_ratio_z * C(cluster_type, Treatment(reference="주거형"))
    + day_night_ratio_z * C(cluster_type, Treatment(reference="주거형"))
    + food_accom_ratio_z * C(cluster_type, Treatment(reference="주거형"))
"""

# ------------------------------------------------------------
# 4) 모델 적합
# ------------------------------------------------------------

model_m1 = smf.ols(formula_m1, data=train_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": train_df["gu"]}
)

model_m2 = smf.ols(formula_m2, data=train_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": train_df["gu"]}
)

model_m3 = smf.ols(formula_m3, data=train_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": train_df["gu"]}
)

# ------------------------------------------------------------
# 5) 성능 비교표 생성
# ------------------------------------------------------------

model_perf = pd.DataFrame({
    "모델": [
        "Model 1",
        "Model 2",
        "Model 3"
    ],
    "구성": [
        "구조 변수만",
        "+ 클러스터 유형",
        "+ 상호작용항"
    ],
    "설명": [
        "인구·산업·생활 변수",
        "도시 유형 차이 반영",
        "유형별 변수 효과 차이 반영"
    ],
    "Adjusted R²": [
        model_m1.rsquared_adj,
        model_m2.rsquared_adj,
        model_m3.rsquared_adj
    ],
    "AIC": [
        model_m1.aic,
        model_m2.aic,
        model_m3.aic
    ],
    "BIC": [
        model_m1.bic,
        model_m2.bic,
        model_m3.bic
    ],
    "N": [
        int(model_m1.nobs),
        int(model_m2.nobs),
        int(model_m3.nobs)
    ]
})

# 보기 좋게 반올림
model_perf_display = model_perf.copy()
model_perf_display["Adjusted R²"] = model_perf_display["Adjusted R²"].round(3)
model_perf_display["AIC"] = model_perf_display["AIC"].round(1)
model_perf_display["BIC"] = model_perf_display["BIC"].round(1)

display(model_perf_display)

# CSV 저장
model_perf_display.to_csv(
    "model_performance_for_ppt.csv",
    index=False,
    encoding="utf-8-sig"
)

,모델,구성,설명,Adjusted R²,AIC,BIC,N
0,Model 1,구조 변수만,인구·산업·생활 변수,0.551,26.9,55.2,97
1,Model 2,+ 클러스터 유형,도시 유형 차이 반영,0.635,9.3,45.4,97
2,Model 3,+ 상호작용항,유형별 변수 효과 차이 반영,0.911,-118.2,-51.2,97
